In [1]:
import pandas as pd
import numpy as np

def preencher_dataset():

    # Carregar o dataset
    df = pd.read_parquet("base_lourival_janelas_moveis_6h.parquet")

    # Ordenar por paciente, internação, UTI e tempo
    df = df.sort_values(
        ["subject_id", "hadm_id", "icustays", "charttime"]
    )

    # Colunas identificadoras, temporais e que não devem ser imputadas
    ignorar = [
        "subject_id",
        "hadm_id",
        "icustays",
        "charttime",
        "tem_sepse",
        "janela_6h_paciente"
    ]

    # Substituir strings vazias por NaN
    df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # Preencher valores faltantes dentro de cada paciente/internação/UTI
    for coluna in df.columns:

        if coluna in ignorar:
            continue

        df[coluna] = (
            df.groupby(
                ["subject_id", "hadm_id", "icustays"]
            )[coluna]
            .transform(lambda x: x.ffill().bfill())
        )

    # Preencher qualquer NaN restante nas demais colunas
    for coluna in df.columns:

        if coluna in ignorar:
            continue

        if df[coluna].isnull().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[coluna]):

            df[coluna] = df[coluna].fillna(df[coluna].median())

        else:

            moda = df[coluna].mode(dropna=True)

            if len(moda) > 0:
                df[coluna] = df[coluna].fillna(moda.iloc[0])
            else:
                df[coluna] = df[coluna].fillna("")

    # Conferência
    print("Total de valores nulos:", df.isnull().sum().sum())
    print("Total de strings vazias:", (df.astype(str).eq("")).sum().sum())

    # Salvar
    df.to_parquet(
        "base_lourival_janelas_moveis_6h_fill.parquet",
        index=False
    )

    print("Arquivo salvo com sucesso!")


preencher_dataset()

Total de valores nulos: 264371
Total de strings vazias: 0
Arquivo salvo com sucesso!
